# Part 1 — Global Localization (Perfect Situation)

## Objective
Show that the relative pose between two robots can be easily computed when both are tracked in a shared reference frame.

### Theory
Given global poses $^O T_A$ and $^O T_B$ from a vicon/GT tracker:
$$
^A T_B = (^O T_A)^{-1} \cdot ^O T_B
$$

---

In [ ]:
# Cell 1: Import & Setup
import sys
sys.path.insert(0, '/workspace/Python/Relative-localisation-tutorial')

import numpy as np
from arp_utility_code.UtilityCode.Manifpy_helper import sgal3_to_se3_matrix 
np.set_printoptions(precision=4, suppress=True)

# Seed for reproducibility
np.random.seed(42)

print('Imports complete')

#### Parameters
---
Configure simulation parameters below:

In [ ]:
# Cell 2: Configuration
from src.config import GlobalLocalizationConfig

config = GlobalLocalizationConfig(
    time=10.0,
    dt=0.1,
)

print(f'Simulation: {int(config.time)}s @ {config.dt}s timestep ({len(np.arange(0, config.time, config.dt))} steps)')

#### Build Scenario
---

In [ ]:
# Cell 3: Build Two-Robot Scenario
from src.trajectories import build_two_robot_scenario

scenario = build_two_robot_scenario(config)
traj_a = scenario['gt_traj_a']
traj_b = scenario['gt_traj_b']
rel_offset = scenario['rel_offset']

print(f'Robot A start: {sgal3_to_se3_matrix(traj_a.X_OR)[:3, 3]}')
print(f'Robot B start: {sgal3_to_se3_matrix(traj_b.X_OR)[:3, 3]}')
print(f'Relative offset (desired): {rel_offset}')

#### Compute & Display Relative Pose
---

In [ ]:
# Cell 4: Perfect Case — Direct Computation
from src.relative_pose import compute_relative_pose

result_perfect = compute_relative_pose(traj_a, traj_b)

# Display first few relative poses
print('Time (s) | Rel Pos [m]                | Rel Rot norm (rad)')
print('-' * 65)
for i in range(0, len(result_perfect.time), max(1, 20)):
    t = result_perfect.time[i]
    pos = result_perfect.rel_pos[i]
    orient_norm = np.linalg.norm(result_perfect.rel_orient[i])
    print(f'{t:6.1f} | [{pos[0]:8.4f}, {pos[1]:8.4f}, {pos[2]:8.4f}]  | {orient_norm:12.6f}')

#### Verify Relative Pose Quality
---

In [ ]:
# Cell 5: Analysis
pos_error_norms = np.linalg.norm(result_perfect.rel_pos - rel_offset, axis=1)
rot_error_norms = np.linalg.norm(result_perfect.rel_orient, axis=1)

print('=== Verification ===')
print(f'Position error max:  {pos_error_norms.max():.2e} m')
print(f'Orientation error:   {rot_error_norms.max():.2e} rad')
print(f'Relative pos norm constant? {np.allclose(np.linalg.norm(result_perfect.rel_pos, axis=1), np.linalg.norm(rel_offset))}')


#### Visualization
---

In [ ]:
# Cell 6: Plot trajectories and overlay relative pose arrows
from matplotlib import pyplot as plt

def extract_positions(traj):
    N = len(traj.X_RRi)
    pos = np.zeros((N, 3))
    for i in range(N):
        G = traj.X_OR.compose(traj.X_RRi[i])
        pos[i] = sgal3_to_se3_matrix(G)[:3, 3]
    return pos

pos_a = extract_positions(traj_a)
pos_b = extract_positions(traj_b)

In [ ]:
# Cell 7: Plot positions
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 3D trajectory
ax_3d = fig.add_subplot(121, projection='3d')
ax_3d.plot(pos_a[:, 0], pos_a[:, 1], pos_a[:, 2], 'b-', linewidth=2, label='Robot A')
ax_3d.plot(pos_b[:, 0], pos_b[:, 1], pos_b[:, 2], 'r--', linewidth=2, label='Robot B')
ax_3d.set_xlabel('X (m)'); ax_3d.set_ylabel('Y (m)'); ax_3d.set_zlabel('Z (m)')
ax_3d.set_title('Global Trajectories (3D)'); ax_3d.legend()

# 2D top-down
ax_2d = axes[1]
ax_2d.plot(pos_a[:, 0], pos_a[:, 1], 'b-', linewidth=2, label='Robot A')
ax_2d.plot(pos_b[:, 0], pos_b[:, 1], 'r--', linewidth=2, label='Robot B')

# Draw periodic arrows showing relative pose
N = len(result_perfect.time)
step = max(1, N // 8)  # draw ~8 arrows
for i in range(0, N, step):
    start = pos_a[i]
    end = pos_b[i]
    ax_2d.arrow(start[0], start[1], end[0]-start[0], end[1]-start[1],
                head_width=0.3, head_length=0.2, fc='green', ec='green', alpha=0.7)

ax_2d.set_title('XY Top-Down with Relative Pose Arrows')
ax_2d.set_xlabel('X (m)'); ax_2d.set_ylabel('Y (m)'); ax_2d.legend(); ax_2d.set_aspect('equal')
plt.tight_layout()
plt.show()

#### Noisy Case — Error Propagation
---

In [ ]:
# Cell 8: Compute with noisy GT
from src.relative_pose import compute_noisy_relative_pose

SIG_POS = 0.01   # meters (1 cm per axis)
SIG_ORI = 0.02   # radians (~1 deg per axis)

# Reset seed for reproducibility in noisy run
np.random.seed(42)
result_noisy = compute_noisy_relative_pose(traj_a, traj_b, sig_pos=SIG_POS, sig_orient=SIG_ORI)

display_noise_error_norms = np.linalg.norm(result_noisy.rel_pos - result_perfect.rel_pos, axis=1)

In [ ]:
# Cell 9: Compare perfect vs noisy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overlay positions
axes[0].plot(pos_a[:, 0], pos_a[:, 1], 'b-', linewidth=2, label='Robot A GT')
axes[0].plot(pos_b[:, 0], pos_b[:, 1], 'r--', linewidth=2, label='Robot B GT')
# Annotate arrows for noisy case
N = len(result_noisy.time)
step = max(1, N // 8)
for i in range(0, N, step):
    T_OA_noisy = sgal3_to_se3_matrix(traj_a.X_OR.compose(traj_a.X_RRi[i]))
    T_OB_noisy = sgal3_to_se3_matrix(traj_b.X_OR.compose(traj_b.X_RRi[i]))
    # Simulate noisy measurement by perturbing positions
    # (This is where you'd normally pull from NoisyGTSensor)
    T_OA_noisy[:3, 3] += np.random.randn(3) * SIG_POS
    T_OB_noisy[:3, 3] += np.random.randn(3) * SIG_POS
    T_AB_noisy = np.linalg.inv(T_OA_noisy) @ T_OB_noisy
    start = T_OA_noisy[:3, 3]
    end = T_OB_noisy[:3, 3]
    axes[0].arrow(start[0], start[1], end[0]-start[0], end[1]-start[1],
                  head_width=0.25, head_length=0.18, fc='orange', ec='orange', alpha=0.6)

axes[0].set_title('Noisy Case: Perturbed Relative Pose')
axes[0].set_xlabel('X (m)'); axes[0].set_ylabel('Y (m)'); axes[0].legend(); axes[0].set_aspect('equal')

# Error magnitude plot
pos_err_perfect = np.linalg.norm(result_perfect.rel_pos - result_noisy.rel_pos, axis=1)
rot_err_perfect = np.linalg.norm(result_perfect.rel_orient - result_noisy.rel_orient, axis=1)

axes[1].plot(result_perfect.time, pos_err_perfect, 'b-', linewidth=1.5, label='Position error')
axes[1].plot(result_perfect.time, rot_err_perfect, 'r-', linewidth=1.5, label='Orientation error (rad * 100)')
axes[1].set_ylabel('Error magnitude'); axes[1].set_xlabel('Time (s)')
axes[1].set_title('Perfect vs Noisy: Error Magnitude'); axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

***
### Summary & Transition to Parts 2–5
- In the perfect case, relative pose computation is exact: $^A T_B$ has constant norm.
- With vicon noise (σ=1 cm pos, σ=0.02 rad orient), errors are small but noticeable.
- Next Part: Analyze how sensor geometry affects GDOP for UWB-based relative localisation when perfect GT is unavailable.